# Notebook 9: Magnitude Sweep on Diff-in-Means Direction

Given coherence recovery of diff-in-means translation for RMU, check if varying magnitude
of translation could potentially restore more capability

In [1]:
# Make `src` importable when running from notebooks/.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from src.magnitude_sweep import (
    run_sweep, save_sweep, load_sweep,
    plot_sweep, plot_per_question_heatmap, plot_coherence_vs_knowledge,
)

from src.config import CHECKPOINTS, RETAIN, BASE_MODEL, checkpoint, layer_name, LIVE_CASES, CONTROLS

# driver-local, stays in the notebook:
OUT_DIR = "../results/sweep_magnitude_rmu.json"
os.makedirs(OUT_DIR, exist_ok=True)

SWEEP_JSON = "../data/sweep_magnitude_rmu.json"
ORACLE_LOGPROB = -2.5     # retain90 honest-ignorance reference from status notes
RMU_BASELINE  = -7.9      # known RMU c=0 anchor; row 1 must reproduce this

## 1. Data + direction

Replace the loader below with however you already load TOFU in your other
notebooks. You need four aligned lists: forget questions, forget gold answers,
and retain questions (retain answers not needed — retain is only used to build
the diff-in-means direction).

## 2a. Run the sweep (skip if loading a precomputed JSON)

`diff_in_means_direction` returns the FIXED axis `d` and the data-derived
`gap`. Only `c` varies in the sweep; `d` is held constant so the result is
attributable to magnitude alone.

**Watch the first printed row:** `c=0` must reproduce RMU's ≈ −7.9 baseline.
If it doesn't, the layer or direction is wrong — stop and fix before waiting
out the full run.

In [ ]:
RUN_SWEEP = True   # set False to skip straight to 2b (load precomputed)

if RUN_SWEEP:
    from src.model_loader import load_model
    from src.intervention import diff_in_means_direction

    model, tok, dev = load_model(
        "open-unlearning/tofu_Llama-3.2-1B-Instruct_RMU"
    )
    d, gap = diff_in_means_direction(
        model, tok, dev, forget_qs, retain_qs,
        layer_name="model.layers.10", return_raw_gap=True,
    )
    print(f"diff-in-means gap = {gap:+.4f}")

    res = run_sweep(
        model, tok, dev,
        prompts=forget_qs, answers=forget_answers,
        layer_name="model.layers.10", direction=d, gap=gap,
        n_steps=11, max_mult=2.0, max_new_tokens=60,
    )
    save_sweep(res, SWEEP_JSON)

    # Correctness anchors — fail loudly if the sweep disagrees with Phase 1.
    c0 = res["rows"][0]["mean_logprob"]
    cgap = next(r for r in res["rows"] if abs(r["c_over_gap"] - 1.0) < 1e-6)
    print(f"\nANCHOR c=0   log-prob = {c0:+.3f}  (expect ≈ {RMU_BASELINE})")
    print(f"ANCHOR c=gap log-prob = {cgap['mean_logprob']:+.3f}  "
          f"(expect ≈ {ORACLE_LOGPROB})")

## 2b. Load a precomputed sweep (if you ran the headless script)

In [ ]:
if not RUN_SWEEP:
    res = load_sweep(SWEEP_JSON)
    print(f"loaded sweep: {res['n_questions']} questions, "
          f"{len(res['rows'])} c-values, gap={res['gap']:+.4f}")

## 3. Thesis chart — coherence climbs, knowledge stays flat

Twin y-axes on a shared `c`. Blue = mean log-prob (coherence). Red = fact-hit
rate (knowledge). Dashed line = c=gap (retain displacement). Dotted = oracle.

In [ ]:
ax = plot_sweep(res, oracle_logprob=ORACLE_LOGPROB)
plt.tight_layout()
plt.savefig("../data/fig_sweep_thesis.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-question heatmap — offset vs selective recovery

Uniform brightening down every row = a constant offset (translation just adds
a constant; nothing selectively recovered). A few bright rows = selective
recovery. Phase-1 read predicts the uniform pattern.

In [ ]:
ax = plot_per_question_heatmap(res)
plt.tight_layout()
plt.savefig("../data/fig_sweep_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Coherence-vs-knowledge scatter — the artifact as one shape

Each point is one `c`. Horizontal smear (knowledge flat while coherence
climbs) = coherence artifact. A diagonal would mean real recovery.

In [ ]:
ax = plot_coherence_vs_knowledge(res)
plt.tight_layout()
plt.savefig("../data/fig_sweep_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Hand-audit a few generations

The fact matcher is a generous screening metric — confirm by eye that flat
fact-hits really are confabulations, not matcher misses. Look at c=gap and
c=2*gap rows: fluent? on-topic? and yet wrong on the discrete fact?

In [ ]:
for r in res["rows"]:
    if abs(r["c_over_gap"] - 1.0) < 1e-6 or abs(r["c_over_gap"] - 2.0) < 1e-6:
        print(f"\n=== c/gap = {r['c_over_gap']:.1f}  "
              f"(log-prob {r['mean_logprob']:+.2f}, "
              f"fact-hit {r['fact_hit_rate']:.2f}) ===")
        for i, g in enumerate(r["sample_generations"]):
            print(f"  [{i}] {g[:160]}")